# 🐛 Crop Pest Infestation — Image Classification Pipeline

**25,220 labeled images** across **22 classes** — pests and diseases affecting Cashew, Cassava, Maize, and Tomato crops.

## Pipeline Overview
1. Dataset Information
2. Import Libraries
3. Data Loading & Exploration
4. Class Distribution Analysis
5. Image Inspection & Quality Check
6. Dataset Splits (train/val/test)
7. Image Preprocessing & Augmentation
8. Save Processed Metadata
9. Model-Ready Data Preparation

---
## 1. Dataset Information

| Field | Value |
|-------|-------|
| **Dataset** | Crop Pest Infestation |
| **Total Images** | 25,220 |
| **Classes** | 22 |
| **Crops** | Cashew, Cassava, Maize, Tomato |
| **Format** | JPEG images |
| **Resolution** | Varies (see analysis below) |
| **Splits** | 70% train / 15% val / 15% test |

### Classes by Crop

| Crop | Classes |
|------|---------|
| Cashew | anthracnose, gumosis, healthy, leaf miner, red rust |
| Cassava | bacterial blight, brown spot, green mite, healthy, mosaic |
| Maize | fall armyworm, grasshoper, healthy, leaf beetle, leaf blight, leaf spot, streak virus |
| Tomato | healthy, leaf blight, leaf curl, septoria leaf spot, verticulium wilt |

---
## 2. Import Libraries

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

---
## 3. Data Loading & Exploration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

# ── Load split CSVs ──────────────────────────────────────────────
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
all_splits = pd.read_csv(PROCESSED_DIR / "all_splits.csv")

with open(PROCESSED_DIR / "class_labels.json") as f:
    class_labels = json.load(f)

print(f"Train: {len(train_df):>6} images")
print(f"Val:   {len(val_df):>6} images")
print(f"Test:  {len(test_df):>6} images")
print(f"Total: {len(all_splits):>6} images")
print(f"Classes: {len(class_labels)}")
print(f"Class map: {class_labels}")

In [ ]:
# ── Quick preview ───────────────────────────────────────────────
train_df.head(10)

---
## 4. Class Distribution Analysis

In [ ]:
# ── Class distribution ──────────────────────────────────────────
dist = all_splits['label'].value_counts()

fig, ax = plt.subplots(figsize=(14, 6))
colors = sns.color_palette("Set2", n_colors=len(dist))
bars = ax.barh(range(len(dist)), dist.values, color=colors)
ax.set_yticks(range(len(dist)))
ax.set_yticklabels(dist.index)
ax.set_xlabel("Image Count")
ax.set_title("Class Distribution — Crop Pest Infestation", fontsize=14, fontweight='bold')

for bar, val in zip(bars, dist.values):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Crop-level distribution ─────────────────────────────────────
crop_dist = all_splits.groupby('crop').size().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3']
ax.pie(crop_dist.values, labels=crop_dist.index, autopct='%1.1f%%',
       colors=colors, startangle=90, explode=[0.03]*4)
ax.set_title("Images by Crop", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

for crop, cnt in crop_dist.items():
    print(f"{crop:10s} → {cnt:>5} images ({100*cnt/len(all_splits):.1f}%)")

In [ ]:
# ── Class distribution per crop ─────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
crops = sorted(all_splits['crop'].unique())

for ax_, crop in zip(axes.flatten(), crops):
    subset = all_splits[all_splits['crop'] == crop]
    dist_ = subset['label'].value_counts()
    ax_.barh(range(len(dist_)), dist_.values, color=sns.color_palette("Set2"))
    ax_.set_yticks(range(len(dist_)))
    ax_.set_yticklabels([c.replace(f'{crop} ', '') for c in dist_.index])
    ax_.set_title(f"{crop} — {len(subset)} images", fontsize=12, fontweight='bold')
    for bar, val in zip(ax_.patches, dist_.values):
        ax_.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## 5. Image Inspection & Quality Check

In [ ]:
# ── Sample images from each class ───────────────────────────────
n_samples = 3
fig, axes = plt.subplots(len(class_labels), n_samples, figsize=(12, 4 * len(class_labels)))

for i, (label_id, class_name) in class_labels.items():
    class_images = all_splits[all_splits['label'] == class_name]['image_path'].values
    samples = random.sample(list(class_images), min(n_samples, len(class_images)))
    
    for j, rel_path in enumerate(samples):
        img = Image.open(RAW_DIR / rel_path)
        axes[int(i)][j].imshow(img)
        axes[int(i)][j].axis('off')
        if j == 0:
            axes[int(i)][j].set_ylabel(class_name, fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Image dimensions analysis ───────────────────────────────────
dims = []
for rel_path in all_splits['image_path'].values[:1000]:  # sample 1000 for speed
    try:
        img = Image.open(RAW_DIR / rel_path)
        dims.append(img.size)
    except:
        pass

widths = [d[0] for d in dims]
heights = [d[1] for d in dims]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.hist(widths, bins=30, color='#66c2a5', edgecolor='white')
ax1.set_title(f"Width Distribution (sample of {len(dims)})", fontweight='bold')
ax1.set_xlabel("Width (px)")
ax1.set_ylabel("Count")

ax2.hist(heights, bins=30, color='#fc8d62', edgecolor='white')
ax2.set_title(f"Height Distribution (sample of {len(dims)})", fontweight='bold')
ax2.set_xlabel("Height (px)")
ax2.set_ylabel("Count")

plt.tight_layout()
plt.show()

print(f"Width range:  {min(widths)} – {max(widths)} px")
print(f"Height range: {min(heights)} – {max(heights)} px")
print(f"Mean size:    {int(np.mean(widths))} x {int(np.mean(heights))} px")

---
## 6. Dataset Splits — Verification

Splits were created by `scripts/01_create_splits.py` using stratified sampling.
Verify each split preserves the original class distribution.

In [ ]:
# ── Verify stratification ───────────────────────────────────────
def split_distribution(df, split_name):
    dist = df['label'].value_counts()
    total = len(df)
    print(f"\n{split_name.upper()} ({total} images):")
    for cls, cnt in dist.items():
        print(f"  {cls:35s} {cnt:5d} ({100*cnt/total:.1f}%)")

split_distribution(train_df, 'train')
split_distribution(val_df, 'val')
split_distribution(test_df, 'test')

In [ ]:
# ── Visual split comparison ─────────────────────────────────────
all_splits['pct'] = all_splits.groupby(['label', 'split']).transform('size') / \
                    all_splits.groupby('label').transform('size') * 100

fig, ax = plt.subplots(figsize=(16, 8))

for i, split in enumerate(['train', 'val', 'test']):
    subset = all_splits[all_splits['split'] == split]
    ax.barh(
        np.arange(len(class_labels)) + i * 0.25,
        subset.groupby('label').size().values,
        height=0.25,
        label=split,
        color=['#66c2a5', '#fc8d62', '#8da0cb'][i]
    )

ax.set_yticks(np.arange(len(class_labels)) + 0.25)
ax.set_yticklabels([class_labels[i] for i in range(len(class_labels))])
ax.set_xlabel("Image Count")
ax.set_title("Split Distribution by Class", fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Image Preprocessing & Augmentation

Recommended preprocessing steps for model training:

In [ ]:
# ── Preprocessing functions ─────────────────────────────────────
from PIL import Image, ImageEnhance


def resize_and_center_crop(img, target_size=(224, 224)):
    """Resize maintaining aspect ratio, then center crop to target."""
    ratio = max(target_size[0] / img.width, target_size[1] / img.height)
    new_size = (int(img.width * ratio), int(img.height * ratio))
    img = img.resize(new_size, Image.LANCZOS)
    
    left = (img.width - target_size[0]) // 2
    top = (img.height - target_size[1]) // 2
    return img.crop((left, top, left + target_size[0], top + target_size[1]))


def preprocess_pipeline(img_path, target_size=(224, 224)):
    """Full preprocessing: load → resize/crop → normalize."""
    img = Image.open(img_path).convert('RGB')
    img = resize_and_center_crop(img, target_size)
    return img


# ── Demo the pipeline on a sample ──────────────────────────────
sample_path = RAW_DIR / train_df.iloc[0]['image_path']
original = Image.open(sample_path)
processed = preprocess_pipeline(sample_path)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(original)
ax1.set_title(f"Original: {original.size}", fontweight='bold')
ax1.axis('off')
ax2.imshow(processed)
ax2.set_title(f"Processed: {processed.size}", fontweight='bold')
ax2.axis('off')
plt.tight_layout()
plt.show()
print(f"Label: {train_df.iloc[0]['label']}")

In [ ]:
# ── Augmentation examples ───────────────────────────────────────
from PIL import ImageFilter

img = Image.open(sample_path).convert('RGB')
img = resize_and_center_crop(img)

augmentations = {
    "Original": img,
    "Horiz. Flip": img.transpose(Image.FLIP_LEFT_RIGHT),
    "Vert. Flip": img.transpose(Image.FLIP_TOP_BOTTOM),
    "Rotated 90": img.rotate(90, expand=True).resize(img.size, Image.LANCZOS),
    "Blurred": img.filter(ImageFilter.GaussianBlur(2)),
    "Brightness +30%": ImageEnhance.Brightness(img).enhance(1.3),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax_, (name, aug_img) in zip(axes.flatten(), augmentations.items()):
    ax_.imshow(aug_img)
    ax_.set_title(name, fontweight='bold')
    ax_.axis('off')
plt.tight_layout()
plt.show()

---
## 8. Save Processed Metadata

In [ ]:
# ── Load existing stats and enhance ──────────────────────────────
with open(PROCESSED_DIR / "dataset_stats.json") as f:
    stats = json.load(f)

stats["preprocessing"] = {
    "resize_method": "center_crop",
    "target_size": "224x224 (recommended)",
    "augmentations": ["horizontal_flip", "vertical_flip", "rotation",
                       "brightness", "contrast", "gaussian_blur"],
    "normalization": "ImageNet stats (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])",
}

with open(PROCESSED_DIR / "dataset_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print("✅ Metadata updated with preprocessing info")

---
## 9. Model-Ready Data Preparation

Generate PyTorch/TensorFlow-ready data loaders.

In [ ]:
# ── Prepare data loader CSVs with absolute paths ────────────────
RAW_ABS = RAW_DIR.resolve()

# Load class_labels as integer mapping
class_to_id = {v: int(k) for k, v in class_labels.items()}

for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    split_df = split_df.copy()
    split_df['image_path_abs'] = split_df['image_path'].apply(
        lambda p: str(RAW_ABS / p)
    )
    split_df['label_id'] = split_df['label'].map(class_to_id)
    out_path = PROCESSED_DIR / f"{split_name}_ml.csv"
    split_df.to_csv(out_path, index=False)
    print(f"✅ {out_path.name}: {len(split_df)} samples")

In [ ]:
# ── Verify ML-ready files ───────────────────────────────────────
ml_train = pd.read_csv(PROCESSED_DIR / 'train_ml.csv')
print(f"Train: {ml_train.shape}")
print(f"Columns: {list(ml_train.columns)}")
print(f"\nFirst 5 samples:")
print(ml_train.head())
print(f"\nClass distribution in training set:")
print(ml_train['label'].value_counts().to_string())

---

## ✅ Summary

| Metric | Value |
|--------|-------|
| Total images | 25,220 |
| Classes | 22 |
| Crops | 4 (Cashew, Cassava, Maize, Tomato) |
| Train | 17,654 (70%) |
| Val | 3,783 (15%) |
| Test | 3,783 (15%) |
| Split method | Stratified (preserves class proportions) |
| Imbalance ratio | 1:13 (most vs least frequent class) |

### Output Files
- `data/processed/train.csv` / `val.csv` / `test.csv` — split CSVs with relative paths
- `data/processed/train_ml.csv` / `val_ml.csv` / `test_ml.csv` — ML-ready with absolute paths + label IDs
- `data/processed/all_splits.csv` — combined dataset
- `data/processed/class_labels.json` — class ID mapping
- `data/processed/dataset_stats.json` — full metadata